## Faster C-NN
___

In [ ]:
import os
from pathlib import Path

import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image
from torchvision.io import read_image
import pandas as pd

#=== Variables varias ===
CARPETA_ENTRADA = "data/FjordFish/images/train"
CARPETA_SALIDA  = Path("resultados")      
ARCHIVO_CSV     = "detecciones.csv" 
UMBRAL_CONFIANZA    = 0.9         
EXTENSION = ".jpg" 


# Creamos la carpeta de salida de las imagenes procesadas
os.makedirs(CARPETA_SALIDA, exist_ok=True)


#=== Cargamos el modelo faster rc-nn, en su version 2 (EXPLICAR EL PQ USAMOS LA VERSION 2 IMPORTANTE) ===

#Usamos los pesos del faster rc-nn que usa los pesos entrenados del dataset COCO :D
pesos = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(
    #Pasamos los pesos al modelo
    weights=pesos,

    #Pasamos el umbral de confianza, con el cual obtendrá las cajas que sobrepasen dicho umbral 
    box_score_thresh=UMBRAL_CONFIANZA,

).to("cpu")
model.eval()

#Obtenemos la transformación de la imagen usada en el entrenamiento
preprocess = pesos.transforms()

#Guardamos las rutas de las imagenes
carpeta = Path(CARPETA_ENTRADA)
rutas = []

# no las procesa en el orden de la carpeta, pero hacer que se guie por ese orden es un cacho asique dejemoslo asi nomas
for archivo in sorted(carpeta.iterdir()):
    rutas.append(archivo)



#=== procesamos las imagenes y guardamos su información ===

#En esta lista van guardados los objetos encontrados, junto con su información más relevante
registros = []  

for ruta in rutas:
    print(f"Procesando: {ruta.name}")

    # leemos la imagen
    img = read_image(str(ruta))

    # Preprocesar y mover al dispositivo
    batch = [preprocess(img).to("cpu")]

    # Desactivamos el calculo de la gradiente para adivinar y no entrenar
    with torch.no_grad():
        prediction = model(batch)[0]


    cajas  = prediction["boxes"].cpu()
    etiqueta = prediction["labels"].cpu()
    confianza = prediction["scores"].cpu()

    # Caso para cuando no se haya encontrado ningún objeto
    if len(cajas) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
        
    # Caso en donde encontramos objetos en las imagenes
    else:
        # Nombres de las categorías
        nombres = [pesos.meta["categories"][i] for i in etiqueta.tolist()]

        # Guardamos en los registros la informacion de cada objeto encontrado
        for nombre, score, box in zip(nombres, confianza.tolist(), cajas.tolist()):
            x1, y1, x2, y2 = box
            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre,
                "confianza": round(score, 4),
                "x1": round(x1, 1), "y1": round(y1, 1),
                "x2": round(x2, 1), "y2": round(y2, 1),
            })

    # Dibujar las cajas sobre la imagen original
    if len(cajas) > 0:
        nombres = [pesos.meta["categories"][i] for i in etiqueta.tolist()]
        img_anotada = draw_bounding_boxes(
            img,
            boxes=cajas,
            labels=nombres,
            colors="red",
            width=4,
            font_size=30,
        )
    else:
        img_anotada = img

    to_pil_image(img_anotada).save(CARPETA_SALIDA / ruta.name)


#=== Creamos el dt y lo exportamos ===
df = pd.DataFrame(registros)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")

Procesando: 1.jpg
